In [51]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad
import joblib

In [52]:
# Load experimental data
atlas_data = pd.read_csv('../../data/data_0_1/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_9531/1715607575.py:2: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



In [53]:

b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25

n_points = 10000



ensemble_parameters = {
    'atlas': {
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=0.9, upper_factor=1.1):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [54]:
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323

def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):

    def integrand(y, x, mg, a1, a2, m2_func, q_val):
        k        = sqrt_s * x
        phi      = 2*np.pi*y
        jacobian = 2*np.pi*sqrt_s
        return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
                  T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian
    def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func, q_val),
                0, 1, n=n_points
            )[0]

    integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
    
    return integral_value

In [55]:
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }
    
    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        integral_value = full_int(mg, a1, a2, m2, q2, sqrt_s)

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [56]:
def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='pl')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='pl')

def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='pl')
chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13)
chi2_total = chi2_7 + chi2_8 + chi2_13
minuit_born = Minuit(
    chi2_total,
    mg = 0.421,
    a1 = 1.517,
    a2 = 2.05,
    eps = 0.0753
)

minuit_born.migrad()
minuit_born.hesse()


/tmp/ipykernel_9531/1414407444.py:15: RuntimeWarning:

overflow encountered in exp

/tmp/ipykernel_9531/1414407444.py:41: RuntimeWarning:

overflow encountered in multiply

/tmp/ipykernel_9531/1414407444.py:68: RuntimeWarning:

overflow encountered in multiply

/tmp/ipykernel_9531/1414407444.py:11: RuntimeWarning:

invalid value encountered in power



┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 208.5 (χ²/ndof = 3.0)      │             Nfcn = 2911              │
│ EDM = 4.21e-05 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ eps  │  0.0689   │  0.0035   │            │            │         │         │       │
│ 1 │ mg   │   0.384   │   0.009   │            │            │         │         │       │
│ 2 │ a1   │   1.97    │   0.06    │            │            │         │         │       │
│ 3 │ a2   │     0     │  0.8e-3   │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────┐
│     │      eps       mg       a1       a2 │
├─────┼─────────────────────────────────────┤
│ eps │ 1.19e-05 0.030e-3 0.184e-3   0.1e-6 │
│  mg │ 0.030e-3 7.72e-05  0.47e-3   0.2e-6 │
│  a1 │ 0.184e-3  0.47e-3  0.00348   1.3e-6 │
│  a2 │   0.1e-6   0.2e-6   1.3e-6  6.2e-07 │
└─────┴─────────────────────────────────────┘

In [47]:
mg_min = 0.421
eps_min = 0.0753
a1_min = 1.517
a2_min = 2.05

In [ ]:
import numpy as np
from scipy.special import j0
from scipy.integrate import quad

lst_chi = []
lst_amp_eik = []
lst_diff_sigma = []

lst_b_integration = np.linspace(0, 10, 50)

sqrt_s = 7000
s = sqrt_s ** 2

eps_rel = 1e-6
eps_abs = 1e-16

q_max = 5

def chi_integrand(q_val, b_val):
    q2_val = q_val ** 2
    t = -q2_val

    diff_t = full_int(
        mg_min,
        a1_min,
        a2_min,
        m2_pl,
        q2_val,
        sqrt_s
    )

    born_amp = amp_calculation(
        diff_t,
        s,
        eps_min,
        t
    )

    return (1/s) * q_val * j0(b_val * q_val) * born_amp


for b_val in lst_b_integration:

    # quad doesn't handle complex integrands natively, so split real and imag
    real_part, _ = quad(lambda q: np.real(chi_integrand(q, b_val)), 0, q_max, epsrel=eps_rel, epsabs=eps_abs)
    imag_part, _ = quad(lambda q: np.imag(chi_integrand(q, b_val)), 0, q_max, epsrel=eps_rel, epsabs=eps_abs)

    chi_sum = real_part + 1j * imag_part

    print(f"b = {b_val:.2f}, χ(b) = {np.imag(chi_sum):.4f}")

    lst_chi.append(chi_sum)

b = 0.00, χ(b) = 12.4977
b = 0.20, χ(b) = 12.4845
b = 0.41, χ(b) = 12.4452
b = 0.61, χ(b) = 12.3799
b = 0.82, χ(b) = 12.2891
b = 1.02, χ(b) = 12.1733
b = 1.22, χ(b) = 12.0331
b = 1.43, χ(b) = 11.8695
b = 1.63, χ(b) = 11.6834
b = 1.84, χ(b) = 11.4758
b = 2.04, χ(b) = 11.2481
b = 2.24, χ(b) = 11.0015
b = 2.45, χ(b) = 10.7374
b = 2.65, χ(b) = 10.4573
b = 2.86, χ(b) = 10.1627
b = 3.06, χ(b) = 9.8552
b = 3.27, χ(b) = 9.5365
b = 3.47, χ(b) = 9.2081
b = 3.67, χ(b) = 8.8719
b = 3.88, χ(b) = 8.5293
b = 4.08, χ(b) = 8.1821
b = 4.29, χ(b) = 7.8319
b = 4.49, χ(b) = 7.4802
b = 4.69, χ(b) = 7.1286
b = 4.90, χ(b) = 6.7785
b = 5.10, χ(b) = 6.4312
b = 5.31, χ(b) = 6.0882
b = 5.51, χ(b) = 5.7506
b = 5.71, χ(b) = 5.4195
b = 5.92, χ(b) = 5.0959
b = 6.12, χ(b) = 4.7809
b = 6.33, χ(b) = 4.4751
b = 6.53, χ(b) = 4.1794
b = 6.73, χ(b) = 3.8943
b = 6.94, χ(b) = 3.6204
b = 7.14, χ(b) = 3.3581
b = 7.35, χ(b) = 3.1076
b = 7.55, χ(b) = 2.8692
b = 7.76, χ(b) = 2.6430
b = 7.96, χ(b) = 2.4291
b = 8.16, χ(b) = 2.2274
b

In [49]:
def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

In [50]:
# ── Step 2: A_eik(s,t) for each data point ───────────────────────────────────
lst_q_data = x_7_atlas  # |t| = q²

chi_interp_re = np.interp(lst_b_integration, lst_b_integration, np.real(lst_chi))
chi_interp_im = np.interp(lst_b_integration, lst_b_integration, np.imag(lst_chi))

def eik_integrand(b_val, q_out):
    chi = np.interp(b_val, lst_b_integration, chi_interp_re) + \
        1j * np.interp(b_val, lst_b_integration, chi_interp_im)
    return b_val * j0(b_val * q_out) * (1 - np.exp(1j * chi))

for q2_val in lst_q_data:
    q_out = np.sqrt(q2_val)

    real_part, _ = fixed_quad(lambda b: np.real(eik_integrand(b, q_out)), 0, lst_b_integration[-1], n=n_points)
    imag_part, _ = fixed_quad(lambda b: np.imag(eik_integrand(b, q_out)), 0, lst_b_integration[-1], n=n_points)

    A_eik = 1j * s * (real_part + 1j * imag_part)
    lst_amp_eik.append(A_eik)

    dsigma = (np.pi / s**2) * np.abs(A_eik)**2 * 0.389379323
    lst_diff_sigma.append(dsigma)
    print(f"q² = {q2_val:.4f}, dσ/dt = {dsigma:.4e}")


# ── Step 3: Plot ──────────────────────────────────────────────────────────────
fig_atlas = go.Figure()

add_differential_trace(fig_atlas, lst_q_data, lst_diff_sigma,
                       label='7 TeV eik', color='blue', mg_model='pl')

add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas,
               name='ATLAS 7 TeV', show_label=True, mode='markers')

fig_atlas.update_layout(
    title='dσ/dt vs. |t| — Eikonal (PL model) vs ATLAS data',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Model',
    plot_bgcolor='white',
    hovermode='x unified'
)
fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')
fig_atlas.show()

q² = 0.0110, dσ/dt = 1.9289e+03
q² = 0.0132, dσ/dt = 1.8279e+03
q² = 0.0160, dσ/dt = 1.7058e+03
q² = 0.0192, dσ/dt = 1.5746e+03
q² = 0.0227, dσ/dt = 1.4407e+03
q² = 0.0265, dσ/dt = 1.3061e+03
q² = 0.0307, dσ/dt = 1.1695e+03
q² = 0.0352, dσ/dt = 1.0364e+03
q² = 0.0400, dσ/dt = 9.0826e+02
q² = 0.0450, dσ/dt = 7.8872e+02
q² = 0.0502, dσ/dt = 6.7817e+02
q² = 0.0559, dσ/dt = 5.7158e+02
q² = 0.0619, dσ/dt = 4.7419e+02
q² = 0.0679, dσ/dt = 3.9035e+02
q² = 0.0744, dσ/dt = 3.1303e+02
q² = 0.0814, dσ/dt = 2.4354e+02
q² = 0.0884, dσ/dt = 1.8643e+02
q² = 0.0959, dσ/dt = 1.3703e+02
q² = 0.1037, dσ/dt = 9.6612e+01
q² = 0.1112, dσ/dt = 6.6597e+01
q² = 0.1194, dσ/dt = 4.2005e+01
q² = 0.1284, dσ/dt = 2.3057e+01
q² = 0.1374, dσ/dt = 1.0783e+01
q² = 0.1468, dσ/dt = 3.4574e+00
q² = 0.1568, dσ/dt = 2.6283e-01
q² = 0.1668, dσ/dt = 4.5256e-01
q² = 0.1768, dσ/dt = 2.9198e+00
q² = 0.1873, dσ/dt = 7.0124e+00
q² = 0.1978, dσ/dt = 1.1885e+01
